## Importing required libraries


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run "/Workspace/Users/abdulm63633@gmail.com/Ecommerce Lakehouse Project/01_setup_file/setup_utils"

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog", "ecommerce_lakehouse_project", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

In [0]:
df_bronze = spark.sql(
    f"""
    SELECT *
    FROM {catalog}.{bronze_schema}.{data_source}
    """
)
df_bronze.show()

In [0]:
# =============================== QUALITY CHECK -===========================

null_counts = df_bronze.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_bronze.columns
])

null_counts.show()

df_bronze.groupBy(
    "order_id",
    "product_id"
).count().filter(
    F.col("count") > 1
).show()

df_bronze.filter(
    (F.col("quantity") <= 0)
    | F.col("quantity").isNull()
).show()

df_bronze.filter(
    F.col("order_timestamp") > F.current_timestamp()
).show()

In [0]:
df_customers = spark.table(
    f"{catalog}.{gold_schema}.dim_customers"
)

df_products = spark.table(
    f"{catalog}.{gold_schema}.dim_products"
)

df_payments = spark.table(
    f"{catalog}.{gold_schema}.fact_payments"
)

df_shipments = spark.table(
    f"{catalog}.{gold_schema}.fact_shipments"
)

In [0]:
df_valid_customer = df_bronze.join( df_customers.select("customer_id"), on="customer_id", how="inner" ) 
df_valid_customer.show()

In [0]:
df_invalid_customer = df_bronze.join(
    df_customers.select("customer_id"),
    on="customer_id",
    how="left_anti"
)
df_invalid_customer.show()

In [0]:
df_silver = df_valid_customer.filter(
    F.col("quantity") > 0
)

In [0]:
df_silver = df_silver.dropDuplicates([
    "order_id",
    "product_id"
])
df_silver = df_silver.filter((F.col("product_id").isNotNull()) & (F.col("order_id").isNotNull()))
df_silver.show()

In [0]:
if not spark.catalog.tableExists(
    f"{catalog}.{silver_schema}.{data_source}"
):

    (
        df_silver.write
            .format("delta")
            .option("delta.enableChangeDataFeed", "true")
            .mode("overwrite")
            .saveAsTable(
                f"{catalog}.{silver_schema}.{data_source}"
            )
    )

    print(
        f"Successfully created Silver table: {data_source}"
    )


# ============================================================
# INCREMENTAL MERGE
# ============================================================

else:

    print(
        f"Running incremental MERGE for {data_source}"
    )

    delta_table = DeltaTable.forName(
        spark,
        f"{catalog}.{silver_schema}.{data_source}"
    )

    (
        delta_table.alias("target")

        .merge(
            source=df_silver.alias("source"),

            condition="""
                target.order_id = source.order_id
                AND
                target.product_id = source.product_id
            """
        )

        .whenMatchedUpdate(

            condition="""
                NOT (target.customer_id <=> source.customer_id)
                OR NOT (target.payment_id <=> source.payment_id)
                OR NOT (target.shipment_id <=> source.shipment_id)
                OR NOT (target.quantity <=> source.quantity)
                OR NOT (target.order_timestamp <=> source.order_timestamp)
            """,

            set={

                "customer_id":
                    "coalesce(source.customer_id, target.customer_id)",

                "payment_id":
                    "coalesce(source.payment_id, target.payment_id)",

                "shipment_id":
                    "coalesce(source.shipment_id, target.shipment_id)",

                "quantity":
                    "coalesce(source.quantity, target.quantity)",

                "order_timestamp":
                    "coalesce(source.order_timestamp, target.order_timestamp)",

            
            }
        )

        .whenNotMatchedInsert(

            values={

                "order_id":
                    "source.order_id",

                "customer_id":
                    "source.customer_id",

                "product_id":
                    "source.product_id",

                "payment_id":
                    "source.payment_id",

                "shipment_id":
                    "source.shipment_id",

                "quantity":
                    "source.quantity",

                "order_timestamp":
                    "source.order_timestamp",

            }
        )

        .execute()
    )

    print(
        f"Successfully merged data into Silver.orders"
    )